# Hands-on Tutorial: Automated Machine Learning with FLAML

This notebook is a **Windows/Anaconda-friendly** counterpart to the auto-sklearn tutorial.

We use **[FLAML](https://github.com/microsoft/FLAML)** (Fast Lightweight AutoML), which is pure-Python + scikit-learn based and generally installs cleanly on Windows.

What you’ll do:
1. Install FLAML
2. Run a **classification** AutoML example (Breast Cancer dataset)
3. Run a **regression** AutoML example (California Housing dataset)
4. Inspect the best model and configuration


## 1) Installation (Windows / Anaconda)

In an Anaconda Prompt (recommended):

```bash
conda create -n flaml python=3.11 -y
conda activate flaml
pip install -U flaml
```

Inside a notebook cell (if you prefer):


In [1]:
!pip -q install -U flaml

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 337.7/337.7 kB 4.4 MB/s eta 0:00:00


## 2) Classification Example

We’ll use the Breast Cancer dataset from scikit-learn and let FLAML search for a good pipeline under a time budget.

Notes:
- `time_budget` is **seconds**.
- You can restrict the search using `estimator_list` (e.g., only `['lgbm', 'rf']`).


In [9]:
import numpy as np
from flaml import AutoML

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, roc_auc_score

# supress the length logs
import logging
logging.getLogger("flaml").setLevel(logging.ERROR)


# Load dataset
X, y = load_breast_cancer(return_X_y=True, as_frame=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42, stratify=y
)

# Fit AutoML
automl_clf = AutoML()
automl_clf.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",
    metric="roc_auc",      # or "accuracy"
    time_budget=60,        # seconds
    n_jobs=-1,
    seed=42,
    verbose=0,
    model_history=True,
)

# Predict
y_pred = automl_clf.predict(X_test)
y_proba = automl_clf.predict_proba(X_test)[:, 1]

print("Best estimator:", automl_clf.best_estimator)
print("Best config:", automl_clf.best_config)
print("Best validation loss:", automl_clf.best_loss)

print("Test accuracy:", accuracy_score(y_test, y_pred))
print("Test ROC-AUC:", roc_auc_score(y_test, y_proba))


Best estimator: lgbm
Best config: {'n_estimators': 27, 'num_leaves': 5, 'min_child_samples': 22, 'learning_rate': 1.0, 'log_max_bin': 6, 'colsample_bytree': np.float64(0.8651396378838615), 'reg_alpha': np.float64(0.017819950629265866), 'reg_lambda': np.float64(0.0871132086709129)}
Best validation loss: 0.007660726764500336
Test accuracy: 0.951048951048951
Test ROC-AUC: 0.9916142557651992


In [11]:
def get_topk_models_by_estimator(automl, k=5):
    """
    Return the top-k estimator families discovered by a fitted FLAML AutoML object,
    along with their best loss, best config, and (optionally) the trained model.

    Notes
    -----
    - For `model` to be available, you must have fit with `model_history=True`.
      Otherwise, `best_model_for_estimator()` may return None or raise.
    - Loss is minimized. For metrics like ROC-AUC, FLAML typically stores loss ≈ (1 - AUC).

    Parameters
    ----------
    automl : flaml.AutoML
        A *fitted* AutoML instance.
    k : int
        Number of top estimators to return.

    Returns
    -------
    topk_models : list[dict]
        Each dict has: {"estimator", "loss", "config", "model"}.
    """
    # Basic validation
    if automl is None:
        raise ValueError("automl is None. Pass a fitted flaml.AutoML instance.")
    if not hasattr(automl, "best_loss_per_estimator"):
        raise AttributeError(
            "This AutoML object does not expose best_loss_per_estimator. "
            "Please check your FLAML version."
        )
    if not hasattr(automl, "best_config_per_estimator"):
        raise AttributeError(
            "This AutoML object does not expose best_config_per_estimator. "
            "Please check your FLAML version."
        )

    loss_by_est = automl.best_loss_per_estimator or {}
    cfg_by_est = automl.best_config_per_estimator or {}

    if len(loss_by_est) == 0:
        raise ValueError(
            "best_loss_per_estimator is empty. "
            "Is the AutoML object fitted successfully?"
        )

    # Clamp k
    k = max(1, int(k))
    k = min(k, len(loss_by_est))

    # Sort estimators by best (lowest) loss
    topk = sorted(loss_by_est.items(), key=lambda kv: kv[1])[:k]

    topk_models = []
    for est, loss in topk:
        cfg = cfg_by_est.get(est, {})

        # Try to fetch the trained model for that estimator (requires model_history=True at fit time)
        model = None
        if hasattr(automl, "best_model_for_estimator"):
            try:
                model = automl.best_model_for_estimator(est)
            except Exception:
                # If model_history wasn't enabled (or model isn't stored), keep model=None
                model = None

        topk_models.append(
            {"estimator": est, "loss": loss, "config": cfg, "model": model}
        )

    return topk_models


In [12]:
topk_models = get_topk_models_by_estimator(automl_clf, k=5)
topk_models

[{'estimator': 'lgbm',
  'loss': np.float64(0.007660726764500336),
  'config': {'n_estimators': 27,
   'num_leaves': 5,
   'min_child_samples': 22,
   'learning_rate': 1.0,
   'log_max_bin': 6,
   'colsample_bytree': np.float64(0.8651396378838615),
   'reg_alpha': np.float64(0.017819950629265866),
   'reg_lambda': np.float64(0.0871132086709129)},
  'model': LGBMEstimator(_estimator_type='classifier',
                colsample_bytree=np.float64(0.8651396378838615),
                learning_rate=1.0, max_bin=63, min_child_samples=22,
                n_estimators=27, n_jobs=-1, num_leaves=5,
                reg_alpha=np.float64(0.017819950629265866),
                reg_lambda=np.float64(0.0871132086709129),
                task=<flaml.automl.task.generic_task.GenericTask object at 0x7cab33884fe0>,
                verbose=-1)},
 {'estimator': 'rf',
  'loss': np.float64(0.009191561844863739),
  'config': {'n_estimators': 33,
   'max_features': 0.1,
   'max_leaves': 9,
   'criterion': np.st

In [15]:
#You can tell the AutoML which estimators to use
# We can stick to those only in SK learn
# Fit AutoML
automl_clf = AutoML()
automl_clf.fit(
    X_train=X_train,
    y_train=y_train,
    task="classification",
    metric="roc_auc",      # or "accuracy"
    time_budget=60,        # seconds
    n_jobs=-1,
    seed=42,
    verbose=0,
    model_history=True,
    estimator_list=["rf", "extra_tree", "lrl1", "lrl2", "svc", "kneighbor"]
)

In [16]:
topk_models = get_topk_models_by_estimator(automl_clf, k=5)
topk_models

[{'estimator': 'extra_tree',
  'loss': np.float64(0.008538609364081063),
  'config': {'n_estimators': 42,
   'max_features': np.float64(0.40178540928054257),
   'max_leaves': 30,
   'criterion': np.str_('gini')},
  'model': ExtraTreesEstimator(_estimator_type='classifier', criterion=np.str_('gini'),
                      max_features=np.float64(0.40178540928054257),
                      max_leaf_nodes=30, n_estimators=42, n_jobs=-1,
                      random_state=12032022,
                      task=<flaml.automl.task.generic_task.GenericTask object at 0x7cab1b74cb30>,
                      verbose=0)},
 {'estimator': 'rf',
  'loss': np.float64(0.009191561844863739),
  'config': {'n_estimators': 33,
   'max_features': 0.1,
   'max_leaves': 9,
   'criterion': np.str_('gini')},
  'model': RandomForestEstimator(_estimator_type='classifier', criterion=np.str_('gini'),
                        max_features=0.1, max_leaf_nodes=9, n_estimators=33,
                        n_jobs=-1, random

## 3) Regression Example

The original notebook used the (now deprecated/removed) Boston Housing dataset.

Here we use **California Housing** (`fetch_california_housing`), which is the standard modern replacement in scikit-learn.


In [19]:
from sklearn.datasets import fetch_california_housing
from sklearn.metrics import mean_squared_error, r2_score

# Load dataset
X, y = fetch_california_housing(return_X_y=True,as_frame=True)

# Split
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.25, random_state=42
)

# Fit AutoML
automl_reg = AutoML()
automl_reg.fit(
    X_train=X_train,
    y_train=y_train,
    task="regression",
    metric="r2",           # or "mse"
    time_budget=60,        # seconds
    n_jobs=-1,
    seed=42,
    verbose=0,
    model_history=True,
)

# Predict
y_pred = automl_reg.predict(X_test)

print("Best estimator:", automl_reg.best_estimator)
print("Best config:", automl_reg.best_config)
print("Best validation loss:", automl_reg.best_loss)

print("Test R2:", r2_score(y_test, y_pred))
print("Test MSE:", mean_squared_error(y_test, y_pred))


Best estimator: lgbm
Best config: {'n_estimators': 3360, 'num_leaves': 4, 'min_child_samples': 45, 'learning_rate': np.float64(0.14834305986929777), 'log_max_bin': 9, 'colsample_bytree': np.float64(0.7937838876066656), 'reg_alpha': np.float64(3.2573687354441763), 'reg_lambda': np.float64(0.13918791251037574)}
Best validation loss: 0.15909800635127008
Test R2: 0.8498204256158528
Test MSE: 0.19872033181324977


In [20]:
topk_reg_models = get_topk_models_by_estimator(automl_reg, k=5)
topk_reg_models

[{'estimator': 'lgbm',
  'loss': 0.15909800635127008,
  'config': {'n_estimators': 3360,
   'num_leaves': 4,
   'min_child_samples': 45,
   'learning_rate': np.float64(0.14834305986929777),
   'log_max_bin': 9,
   'colsample_bytree': np.float64(0.7937838876066656),
   'reg_alpha': np.float64(3.2573687354441763),
   'reg_lambda': np.float64(0.13918791251037574)},
  'model': LGBMEstimator(_estimator_type='regressor',
                colsample_bytree=np.float64(0.7937838876066656),
                learning_rate=np.float64(0.14834305986929777), max_bin=511,
                min_child_samples=45, n_estimators=3360, n_jobs=-1, num_leaves=4,
                reg_alpha=np.float64(3.2573687354441763),
                reg_lambda=np.float64(0.13918791251037574),
                task=<flaml.automl.task.generic_task.GenericTask object at 0x7cab67f400e0>,
                verbose=-1)},
 {'estimator': 'xgboost',
  'loss': 0.18121367418576265,
  'config': {'n_estimators': 43,
   'max_leaves': 69,
   'min

## 4) Your task: Practice hyperparameter optimization (HPO) using FLAML while keeping the model choice fixed to XGBoost.

### Rules

You must use `FLAML`’s AutoML() with:

`estimator_list`=["xgboost"] (XGBoost only — no other estimators allowed)

`metric`="roc_auc" (fixed)

You are allowed to change only AutoML/HPO settings, such as:

`time_budget`

`hpo_method` (e.g., "random", "bs", "auto")

`eval_method` (e.g., "holdout" vs "cv")

`n_splits` (if using `CV`) or `split_ratio` (if using `holdout`)

`seed` (optional, but report it if changed)

### What you must do

Run three experiments under the same dataset split (X_train/X_test/y_train/y_test):

#### Experiment A (Baseline):
time_budget = 30 seconds, hpo_method="random", eval_method="holdout"

#### Experiment B (Smarter search):
time_budget = 30 seconds, hpo_method="bs", eval_method="holdout"

#### Experiment C (More budget + stronger evaluation):
time_budget = 120 seconds, hpo_method="bs", eval_method="cv" with n_splits=5

#### Deliverables (what you submit)

#### A small results table with one row per experiment containing:

time_budget

hpo_method

eval_method

best_config (print automl.best_config)

test ROC-AUC (evaluate on X_test)

#### A short reflection (5–8 lines) answering:

Which experiment performed best on test ROC-AUC?

Did the best configuration change significantly between experiments?

Why do you think CV + larger budget helped (or didn’t help)?

#### Notes / Reminders

You are not tuning the “algorithm choice” — it is fixed to XGBoost.
You are tuning hyperparameters and the HPO strategy.

